In [ ]:
import torch 
import torch.nn as nn
import torch.nn.functional as F 
import math






def scaled_dot_product_attention(Q,K,V,mask = None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q,K.transpose(-2,-1))
    scores = scores / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    weigths = F.softmax(scores,dim=-1)
    return torch.matmul(weigths,V), weigths






class CausalMultiheadattention(nn.Module):
    def __init__(self,d_model,num_heads,dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_K = d_model // num_heads
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model,d_model)
        self.W_V = nn.Linear(d_model,d_model)
        self.W_O = nn.Linear(d_model,d_model)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self,x,batch):
        x = x.view(batch,-1,self.num_heads,self.d_k)
        return x.transpose(1,2)
    def forward(self,x,mask=None):
        batch = x.size(0)
        Q = self.split_heads(self.W_Q(x), batch)
        K = self.split_heads(self.W_K(x), batch)
        V = self.split_heads(self.W_V(x), batch)
        out, _ = scaled_dot_product_attention(Q,K,V,mask)
        out = out.transpose(1,2).contiguous()
        out = out.view(batch,-1,self.d_model)
        return self.W_O(out)
    






class FeedForward(nn.Module):
    def __init__(self,d_model,d_ff,dropout= 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
            

        )

    def forward(self,x):
        return self.net(x)
    







class gptblock(nn.Module):
    def __init__(self,d_model,num_heads,d_ff,dropout=0.1):
        super().__init__()
        self.attention = CausalMultiheadattention(d_model,num_heads,dropout)
        self.ff = FeedForward(d_model,d_ff,dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        def forward(self,x,mask=None):
            x = x + self.attention(self.norm1(x),mask)
            x = x + self.ff(self.norm2(x))
            return x
        







class minigpt(nn.Module):
    def __init__(self,vocab_size ,
                 d_model = 256,
                  num_heads = 8,
                   num_layers = 6,
                    d_ff = 1024,
                     max_len = 1024,
                      dropout = 0.1 ):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size,d_model)
        self.pos_embedding = nn.Embedding(max_len,d_model)
        self.dropout = nn.Dropout(dropout)
        self.block = nn.ModuleList([
            gptblock(d_model,num_heads,d_ff,dropout)
            for _ in range(num_layers)

        ])
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model,vocab_size,bias= False)
        self.apply(self._init_weigths)
    def _init_weights(self,module):
        if isinstance(module,nn.Linear):
            nn.init.normal_(module.weight, mean = 0.0,std=0.03)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
            elif isinstance(module,nn.Linear):
                nn.init.normal_(module.weight,mean=0.0,std=0.3)
    def forward(self,idx,targets=None):
        batch, seq_len = idx.shape
        device = idx.device
        token_emb = self.token_embedding(idx)
        pos = torch.arange(seq_len,device=device)
        pos_Emb = self.


